# 11 - Read Experiment Outputs From Google Drive

This notebook does not train any model. It only reads saved experiment outputs from Google Drive and summarizes metrics for the report.

## 1. Mount Google Drive

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image, Markdown

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('Drive mount skipped:', e)

## 2. Edit Result Paths

Set `RESULTS_ROOT` to the folder that contains one subfolder per experiment, for example `resnet18_covidqu` or `vit_s16_imagenet_covidqu_syn`.

In [ ]:
# Main result folder used by the training notebooks.
RESULTS_ROOT = Path('/content/drive/MyDrive/medcls_cvproject/results/experiments')

# Optional older baseline folder. Leave as None if not needed.
EXTRA_RESULTS_ROOTS = [
    Path('/content/drive/MyDrive/medcls_cvproject/outputs_script_first/results/experiments'),
]

EXPERIMENT_ORDER = [
    'resnet18_none',
    'resnet18_imagenet',
    'resnet18_covidqu',
    'resnet18_imagenet_covidqu',
    'resnet18_covidqu_syn',
    'resnet18_imagenet_covidqu_syn',
    'vit_s16_none',
    'vit_s16_imagenet',
    'vit_s16_covidqu',
    'vit_s16_imagenet_covidqu',
    'vit_s16_covidqu_syn',
    'vit_s16_imagenet_covidqu_syn',
]

print('RESULTS_ROOT:', RESULTS_ROOT, 'exists=', RESULTS_ROOT.exists())
for root in EXTRA_RESULTS_ROOTS:
    print('EXTRA_RESULTS_ROOT:', root, 'exists=', root.exists())

## 3. Find Available Experiment Folders

In [ ]:
def existing_roots():
    roots = []
    if RESULTS_ROOT.exists():
        roots.append(RESULTS_ROOT)
    for root in EXTRA_RESULTS_ROOTS:
        if root is not None and root.exists() and root not in roots:
            roots.append(root)
    return roots

def find_experiment_dir(exp_id):
    for root in existing_roots():
        candidate = root / exp_id
        if candidate.exists():
            return candidate
    return None

available = []
missing = []
for exp_id in EXPERIMENT_ORDER:
    exp_dir = find_experiment_dir(exp_id)
    if exp_dir is None:
        missing.append(exp_id)
    else:
        available.append((exp_id, exp_dir))

print('Available experiments:', len(available))
for exp_id, exp_dir in available:
    print('OK', exp_id, '->', exp_dir)

print('\nMissing experiments:', len(missing))
for exp_id in missing:
    print('MISSING', exp_id)

## 4. Build Metrics Table

In [ ]:
METRIC_KEYS = [
    'accuracy',
    'precision_macro',
    'recall_macro',
    'f1_macro',
    'precision_weighted',
    'recall_weighted',
    'f1_weighted',
    'best_epoch',
    'best_val_f1_macro',
]

def read_json(path):
    if not path.exists():
        return None
    return json.loads(path.read_text())

rows = []
for exp_id in EXPERIMENT_ORDER:
    exp_dir = find_experiment_dir(exp_id)
    row = {'experiment_id': exp_id, 'status': 'missing', 'experiment_dir': None}
    if exp_dir is not None:
        row['status'] = 'found'
        row['experiment_dir'] = str(exp_dir)
        metrics = read_json(exp_dir / 'metrics.json')
        if metrics is None:
            row['status'] = 'missing metrics.json'
        else:
            for key in METRIC_KEYS:
                row[key] = metrics.get(key)
    rows.append(row)

metrics_df = pd.DataFrame(rows)
display(metrics_df)

report_cols = ['experiment_id', 'accuracy', 'precision_macro', 'recall_macro', 'f1_macro', 'best_epoch', 'best_val_f1_macro']
report_df = metrics_df[report_cols].copy()
for col in ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro', 'best_val_f1_macro']:
    report_df[col] = pd.to_numeric(report_df[col], errors='coerce').round(4)
display(report_df)

## 5. Export Summary CSV

This creates a compact table that can be copied into the report.

In [ ]:
SUMMARY_OUT = RESULTS_ROOT.parent / 'experiment_metrics_summary.csv'
SUMMARY_OUT.parent.mkdir(parents=True, exist_ok=True)
report_df.to_csv(SUMMARY_OUT, index=False)
print('Saved:', SUMMARY_OUT)
display(report_df)

## 6. Read Classification Reports

Use `SELECTED_EXPERIMENT` to inspect per-class precision, recall, and F1-score.

In [ ]:
SELECTED_EXPERIMENT = 'resnet18_covidqu'

exp_dir = find_experiment_dir(SELECTED_EXPERIMENT)
if exp_dir is None:
    print('Experiment folder not found:', SELECTED_EXPERIMENT)
else:
    report_path = exp_dir / 'classification_report.csv'
    print('Report path:', report_path, 'exists=', report_path.exists())
    if report_path.exists():
        display(pd.read_csv(report_path))

## 7. Display Confusion Matrices

In [ ]:
for exp_id in EXPERIMENT_ORDER:
    exp_dir = find_experiment_dir(exp_id)
    if exp_dir is None:
        continue
    cm_path = exp_dir / 'confusion_matrix.png'
    if cm_path.exists():
        display(Markdown(f'### {exp_id}'))
        display(Image(filename=str(cm_path)))

## 8. Check Pretraining Checkpoints and Resolved Configs

In [ ]:
checkpoint_rows = []
for exp_id in EXPERIMENT_ORDER:
    exp_dir = find_experiment_dir(exp_id)
    if exp_dir is None:
        continue
    candidates = [
        exp_dir / 'best_checkpoint.pth',
        exp_dir / 'pretrain/checkpoints/best_simclr_backbone.pth',
        exp_dir / 'pretrain/checkpoints/last_simclr_checkpoint.pth',
        exp_dir / 'pretrain/checkpoints/best_dino_teacher.pth',
        exp_dir / 'pretrain/checkpoints/last_dino_checkpoint.pth',
    ]
    row = {'experiment_id': exp_id, 'experiment_dir': str(exp_dir)}
    for path in candidates:
        row[path.name] = path.exists()
    row['config_resolved'] = (exp_dir / 'config_resolved.yaml').exists()
    row['config_resolved_simclr'] = (exp_dir / 'config_resolved_simclr.yaml').exists()
    row['config_resolved_dino'] = (exp_dir / 'config_resolved_dino.yaml').exists()
    checkpoint_rows.append(row)

display(pd.DataFrame(checkpoint_rows))